# 13 — Roster guard & keys

NB 02 covered the static-roster case: every agent ships with the
swarm's pubkey list, ECDSA-verifies every envelope at the receiver,
and drops anything that doesn't match. This notebook handles the
**mid-mission** case:

- A compromised peer's key is **revoked**. Subsequent envelopes from
  that peer fail verification at every receiver.
- A new peer is **admitted**. Receivers reject its envelopes until
  they've received the roster update (the "before-update" window).
- A bad key on the roster — a peer holding a key that no longer
  matches its identity — fails ECDSA and never reaches the trust
  evaluator.

This is L13 in the Workshop curriculum. The attack model: even one
unauthorized signer can poison the trust layer if the envelope path
doesn't gate them out *before* observations are recorded.


## Operator question

**Why is roster integrity a *separate* layer from reputation?**

Reputation is a soft signal — it weights, doesn't reject. The roster
is a hard signal: signatures verify or they don't, full stop. Without
a roster, *any* claim of identity could feed into reputation, and an
attacker who's never been on the swarm could start *gaining* trust
just by talking. The roster keeps Beta(α,β) anchored to peers the
operator has explicitly admitted.

ADR 0009 (hardware attestation) and ADR 0010 (key lifecycle) cover
the mechanism specter-1 uses to bind a key to a robot; this notebook
focuses on what happens when that binding breaks at runtime.


In [1]:
%matplotlib inline
import os
from pathlib import Path
_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(_root)

from specter.crypto import Keypair
from specter.secure_bus import (
    Identity, Roster, ReplayWindow,
    open_envelope, VerificationError,
)
from specter.messages import KIND_OBSERVATION, Observation, encode


## Intuition — the four cases

| Scenario | Sender on roster? | Sig verifies? | Outcome |
|---|---|---|---|
| Honest peer | yes | yes | accepted, observations recorded |
| Revoked compromised peer | no (post-revocation) | yes (still owns its key) | rejected: not on roster |
| New peer pre-roster-update | no (yet) | yes | rejected at receiver |
| Bad key on roster | yes | no (key doesn't match identity) | rejected: ECDSA fails |

We synthesise the first three with one helper.


In [2]:
def try_open(identity: Identity, roster: Roster, ts: int) -> bool:
    """Build an envelope for an observation, seal it with `identity`,
    then open it against `roster`. Returns True iff accepted."""
    obs = Observation(
        observer_id=identity.agent_id,
        subject_id="A1",
        range_m=4.2,
        bearing_rad=0.0,
        timestamp_ns=ts,
    )
    env = identity.seal(KIND_OBSERVATION, encode(obs), timestamp_ns=ts)
    try:
        open_envelope(env, roster, ReplayWindow())
        return True
    except VerificationError:
        return False

# Setup: honest swarm of A0..A3, each with its own keypair on the roster.
identities = {f"A{i}": Identity(f"A{i}", Keypair.generate()) for i in range(4)}
roster = Roster()
for aid, ident in identities.items():
    roster.add(aid, ident.keypair.public_bytes)

# Case A — honest peer admitted.
ok_honest = try_open(identities["A0"], roster, ts=1_000_000_000)
print(f"honest A0:                  accepted = {ok_honest}")


honest A0:                  accepted = True


In [3]:
# Case B — A0 compromised. Operator revokes A0's key. Same envelope
# now hits a roster without A0; expect rejection.
roster_after_revoke = Roster()
for aid in ("A1", "A2", "A3"):
    roster_after_revoke.add(aid, identities[aid].keypair.public_bytes)
ok_revoked = try_open(identities["A0"], roster_after_revoke, ts=2_000_000_000)
print(f"A0 after revocation:        accepted = {ok_revoked}")

# Case C — new peer A4 generated, but the roster hasn't been updated yet.
a4 = Identity("A4", Keypair.generate())
ok_pre_update = try_open(a4, roster, ts=3_000_000_000)
print(f"A4 pre roster-update:       accepted = {ok_pre_update}")

# Case D — A0 is on the roster, but holds the OLD privkey while the
# operator just published a rotated pubkey. ECDSA verify fails because
# the signature was made by the old private key, but the roster only
# knows the new public key.
a0_rotated = Identity("A0", Keypair.generate())  # new keypair, same id
roster_rotated = Roster()
roster_rotated.add("A0", a0_rotated.keypair.public_bytes)  # new key on roster
for aid in ("A1", "A2", "A3"):
    roster_rotated.add(aid, identities[aid].keypair.public_bytes)
ok_bad_key = try_open(identities["A0"], roster_rotated, ts=4_000_000_000)
print(f"A0 bad key (post-rotation): accepted = {ok_bad_key}")


A0 after revocation:        accepted = False
A4 pre roster-update:       accepted = False
A0 bad key (post-rotation): accepted = False


## Claim — the bus rejects unauthorized senders before the trust layer sees them

Each of the three failure cases above asserts to `False`. The
trust-layer evaluator never sees these observations — they're dropped
at the secure-bus boundary. Compare to
`tests/eval/test_identity_attacks.py::test_revoked_compromised_key_rejected_post_revocation`
which gates the runtime claim.


In [4]:
assert ok_honest, "honest envelope should be accepted"
assert not ok_revoked, "revoked-peer envelope should be rejected"
assert not ok_pre_update, "pre-roster-update envelope should be rejected"
assert not ok_bad_key, "bad-key envelope should fail ECDSA verify"
print("OK · honest accepted; revoked / pre-update / bad-key all rejected at the bus")


OK · honest accepted; revoked / pre-update / bad-key all rejected at the bus


## Limit — roster distribution is itself a trust problem

The envelope-verify boundary is only as strong as the operator's
roster-update channel:

- **Race conditions.** Between a compromise being detected and the
  revocation reaching every peer, the attacker's envelopes still
  verify. `tests/eval/test_identity_attacks.py::test_revocation_does_not_affect_other_peers`
  bounds local consistency; cross-peer consistency depends on the
  out-of-band distribution channel.
- **Roster compromise.** If the operator's signing key for the roster
  itself is stolen, every receiver can be silently told to admit a
  Sybil. ADR 0010 traces specter-1's intended TPM/ATECC608A-anchored
  flow for the hardware phase.
- **Attestation gap.** A robot can hold a *valid* key while its
  *behaviour* has been compromised at the application layer (e.g.
  malware running on a real robot). That's a reputation problem, not
  a roster problem — covered in NB 03 / NB 09.

**Pointers:**
- ADR 0001 — signed-envelope format + replay window.
- ADR 0009 — hardware attestation interface.
- ADR 0010 — key lifecycle (rotation, revocation, admission).
- L13 Workshop canvas — visual demonstrator of the bad-key scenario.
